# AlphaForge JS ↔ Python Parity Check

Interactive comparison of JS frontend and Python backend outputs.
Run the JS values in a browser console, paste them here, and compare.

In [ ]:
import numpy as np
from alphaforge.prng import Mulberry32, hash_string
from alphaforge.data import generate_dataset, generate_prices
from alphaforge.scoring import compute_factor_scores_js
from alphaforge.backtest import BacktestConfig, run_backtest
from alphaforge.factors import JS_FACTOR_NAMES

## 1. PRNG Parity

JS console: `const rng = mulberry32(42); for (let i=0; i<10; i++) console.log(rng());`

In [ ]:
# JS reference values (from Node.js)
js_prng = [
    0.6011037519201636,
    0.4482905589975417,
    0.8524657934904099,
    0.6697340414393693,
    0.1748138987459242,
    0.5265925421845168,
    0.2732279943302274,
    0.6247446539346129,
    0.8654746483080089,
    0.4723170551005751,
]

rng = Mulberry32(42)
py_prng = [rng() for _ in range(10)]

print(f"{'Call':<6} {'JS':>22} {'Python':>22} {'Match':>8}")
print("-" * 60)
for i, (js, py) in enumerate(zip(js_prng, py_prng)):
    match = abs(js - py) < 1e-15
    print(f"{i+1:<6} {js:>22.16f} {py:>22.16f} {'✓' if match else '✗':>8}")

## 2. Price Series Parity

JS console:
```js
const d = generateDataset('Technology', 252, 42);
Object.entries(d).forEach(([t, v]) => console.log(t, v.prices[0], v.prices[100], v.prices[252]));
```

In [ ]:
# JS reference values (from Node.js)
js_prices = {
    'AAPL': (272.4509789026, 127.1792147717, 115.7769236105),
    'MSFT': (459.8082464421, 510.6839751874, 575.7069400909),
    'NVDA': (284.1558364569, 227.4812154298, 110.6563871042),
    'GOOGL': (427.8205738752, 402.2571831897, 352.7386412915),
    'META': (415.7345580752, 359.6728975888, 309.1990369116),
    'AVGO': (371.1846341728, 325.6695430838, 337.4395260851),
}

dataset = generate_dataset('Technology', 252, 42)

print(f"{'Ticker':<8} {'Index':>6} {'JS':>16} {'Python':>16} {'Diff':>14}")
print("-" * 65)
for ticker, (p0, p100, p252) in js_prices.items():
    py = dataset[ticker].prices
    for idx, js_val in [(0, p0), (100, p100), (252, p252)]:
        diff = abs(js_val - py[idx])
        print(f"{ticker:<8} {idx:>6} {js_val:>16.10f} {py[idx]:>16.10f} {diff:>14.2e}")

## 3. Factor Scores Parity

In [ ]:
scores = compute_factor_scores_js(dataset, 252)

print(f"{'Ticker':<8}", end="")
for f in JS_FACTOR_NAMES:
    print(f"{f:>18}", end="")
print(f"{'Composite':>12} {'Signal':>10}")
print("-" * 110)

for ticker in sorted(scores.keys()):
    s = scores[ticker]
    print(f"{ticker:<8}", end="")
    for f in JS_FACTOR_NAMES:
        print(f"{s[f]:>18.4f}", end="")
    print(f"{s['_composite']:>12.2f} {s['_signal']:>10}")

## 4. Backtest Parity

In [ ]:
result = run_backtest(BacktestConfig(
    sector='Technology',
    lookback=252,
    factor_name='Momentum (12-1)',
))

m = result.metrics
print(f"Sharpe:       {m.sharpe:.4f}")
print(f"Total Return: {m.total_return*100:.2f}%")
print(f"Max Drawdown: {m.max_dd*100:.2f}%")
print(f"Win Rate:     {m.win_rate*100:.1f}%")
print(f"NAV length:   {len(result.nav)}")
print(f"Final NAV:    {result.nav[-1]:.4f}")